# BPIC 2017 Process Analysis

This notebook examines the BPIC 2017 event log, checks its structure and domain integrity, and discovers and evaluates process models.

**Objective.** Characterize the event log, compare full and `complete` lifecycle traces, validate application and offer outcomes, then discover and compare Petri-net models on cases whose final application status is `A_Pending`, `A_Cancelled`, or `A_Denied`.

**AI Assistance Disclosure.** AI assistance was used to refactor and format the existing analysis, organize its workflow, and document the code. Historical results from the source notebook are retained and labeled; metrics are recalculated when this notebook is run.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pm4py
from IPython.display import display, Image, Markdown

current_directory = Path.cwd().resolve()
project_candidates = [
    current_directory,
    current_directory.parent,
    current_directory / "assigment-1",
    current_directory.parent / "assigment-1",
]
project_root = next(
    (
        candidate
        for candidate in project_candidates
        if (candidate / "src" / "bpic2017_analysis.ipynb").exists()
    ),
    current_directory.parent if current_directory.name == "src" else current_directory,
)
NOTEBOOK_DIR = project_root / "src"
DATA_DIR = project_root / "data"
XES_PATH = DATA_DIR / "BPI2017.xes"
PARQUET_PATH = DATA_DIR / "BPI2017.parquet"
BPMN_DIR = DATA_DIR / "bpmn"
EXTERNAL_BPMN_DIR = BPMN_DIR / "external"
DISCOVERED_BPMN_DIR = BPMN_DIR / "discovered"
CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"
LIFECYCLE = "lifecycle:transition"
SOURCE_ORDER = "_source_event_order"

EXTERNAL_BPMN_DIR.mkdir(parents=True, exist_ok=True)
DISCOVERED_BPMN_DIR.mkdir(parents=True, exist_ok=True)

if not XES_PATH.exists():
    raise FileNotFoundError(f"Place the BPIC 2017 event log at {XES_PATH}")

use_cache = PARQUET_PATH.exists() and PARQUET_PATH.stat().st_mtime >= XES_PATH.stat().st_mtime
if use_cache:
    events = pd.read_parquet(PARQUET_PATH)
    if SOURCE_ORDER not in events.columns:
        raise ValueError(
            f"The parquet cache is missing {SOURCE_ORDER}; remove {PARQUET_PATH} and rerun."
        )
    print(f"Loaded parquet cache: {PARQUET_PATH}")
else:
    event_log = pm4py.read_xes(str(XES_PATH))
    events = pm4py.convert_to_dataframe(event_log)
    events[SOURCE_ORDER] = np.arange(len(events), dtype=np.int64)
    events.to_parquet(PARQUET_PATH, index=False)
    print(f"Imported XES and wrote parquet cache: {PARQUET_PATH}")

required_columns = {CASE_ID, ACTIVITY, TIMESTAMP, LIFECYCLE}
missing_columns = required_columns.difference(events.columns)
if missing_columns:
    raise ValueError(f"Event log is missing required columns: {sorted(missing_columns)}")

events[TIMESTAMP] = pd.to_datetime(events[TIMESTAMP], utc=True, errors="coerce")
events = events.sort_values([CASE_ID, TIMESTAMP, SOURCE_ORDER], kind="mergesort").reset_index(drop=True)
log = events

display(
    pd.DataFrame(
        [
            ("Source file", str(XES_PATH)),
            ("Parquet cache", str(PARQUET_PATH)),
            ("Cache used", use_cache),
            ("Events", f"{len(log):,}"),
            ("Cases", f"{log[CASE_ID].nunique():,}"),
            ("Columns", len(log.columns) - 1),
            ("PM4Py version", pm4py.__version__),
        ],
        columns=["Loading check", "Value"],
    )
)
display(log.drop(columns=[SOURCE_ORDER]).head())


In [ ]:
# Descriptive statistics and event-log complexity
data_columns = [column for column in log.columns if column != SOURCE_ORDER]
case_attributes = [column for column in data_columns if column.startswith("case:")]
event_attributes = [column for column in data_columns if not column.startswith("case:")]
case_lengths = log.groupby(CASE_ID, sort=False).size()
case_bounds = log.groupby(CASE_ID, sort=False)[TIMESTAMP].agg(["min", "max"])
case_durations_days = (case_bounds["max"] - case_bounds["min"]).dt.total_seconds() / 86400
case_durations_days = case_durations_days.dropna()
case_variants = log.groupby(CASE_ID, sort=False)[ACTIVITY].agg(tuple)
variant_counts = case_variants.value_counts()
previous_activity = log.groupby(CASE_ID, sort=False)[ACTIVITY].shift()
directly_follows = previous_activity.notna() & log[ACTIVITY].notna()
self_loop_count = (log[ACTIVITY].eq(previous_activity) & directly_follows).sum()
self_loop_ratio = self_loop_count / directly_follows.sum() if directly_follows.sum() else 0.0
singleton_variant_count = int((variant_counts == 1).sum())
singleton_case_count = int(variant_counts.loc[variant_counts == 1].sum())
categorical_event_attributes = log[event_attributes].select_dtypes(include=["object", "bool"]).columns

descriptive_metrics = pd.DataFrame(
    [
        ("Cases", len(case_lengths)),
        ("Events", len(log)),
        ("Process variants", len(variant_counts)),
        ("Case attribute labels", len(case_attributes)),
        ("Event attribute labels", len(event_attributes)),
        ("Categorical event attributes", len(categorical_event_attributes)),
        ("Case length (events), mean ± SD", f"{case_lengths.mean():.2f} ± {case_lengths.std():.2f}"),
        ("Case length (events), min / median / max", f"{case_lengths.min()} / {case_lengths.median():.0f} / {case_lengths.max()}"),
        (
            "Case duration (days), mean ± SD",
            f"{case_durations_days.mean():.2f} ± {case_durations_days.std():.2f}",
        ),
        (
            "Case duration (days), min / median / max",
            f"{case_durations_days.min():.2f} / {case_durations_days.median():.2f} / {case_durations_days.max():.2f}",
        ),
        ("Singleton variants / all variants", f"{singleton_variant_count / len(variant_counts):.2%}"),
        ("Cases in singleton variants / all cases", f"{singleton_case_count / len(case_lengths):.2%}"),
        ("Self-loop ratio among directly-follows pairs", f"{self_loop_ratio:.2%}"),
    ],
    columns=["Metric", "Value"],
)
display(descriptive_metrics)

top_activities = log[ACTIVITY].value_counts().head(15).sort_values()
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].hist(case_lengths, bins=40, color="#4472C4", edgecolor="white")
axes[0].set(title="Case length distribution", xlabel="Events per case", ylabel="Cases")
axes[1].hist(case_durations_days, bins=40, color="#70AD47", edgecolor="white")
axes[1].set(title="Case duration distribution", xlabel="Duration (days)", ylabel="Cases")
top_activities.plot.barh(ax=axes[2], color="#ED7D31")
axes[2].set(title="15 most frequent activities", xlabel="Events", ylabel="Activity")
fig.tight_layout()
plt.show()


In [ ]:
# Lifecycle granularity and a representative median-length case
median_length = case_lengths.median()
median_case_id = (case_lengths - median_length).abs().sort_values(kind="mergesort").index[0]
full_median_case_trace = log.loc[log[CASE_ID].eq(median_case_id)].copy()
complete_median_case_trace = full_median_case_trace.loc[
    full_median_case_trace[LIFECYCLE].astype("string").str.casefold().eq("complete")
].copy()
trace_columns = [TIMESTAMP, ACTIVITY, LIFECYCLE]
for optional_column in ["Action", "org:resource"]:
    if optional_column in log.columns:
        trace_columns.append(optional_column)

display(
    Markdown(
        f"**Representative case:** `{median_case_id}` — "
        f"{len(full_median_case_trace)} events (median case length: {median_length:.0f})."
    )
)
display(Markdown("#### Full trace (all lifecycle transitions)"))
display(full_median_case_trace[trace_columns].reset_index(drop=True))
display(Markdown("#### Reduced trace (`complete` transitions only)"))
display(complete_median_case_trace[trace_columns].reset_index(drop=True))

complete_duration = (
    (complete_median_case_trace[TIMESTAMP].max() - complete_median_case_trace[TIMESTAMP].min()).total_seconds() / 86400
    if not complete_median_case_trace.empty
    else np.nan
)
trace_comparison = pd.DataFrame(
    [
        ("Events", len(full_median_case_trace), len(complete_median_case_trace)),
        ("Distinct activities", full_median_case_trace[ACTIVITY].nunique(), complete_median_case_trace[ACTIVITY].nunique()),
        (
            "Duration (days)",
            (full_median_case_trace[TIMESTAMP].max() - full_median_case_trace[TIMESTAMP].min()).total_seconds() / 86400,
            complete_duration,
        ),
    ],
    columns=["Measure", "Full trace", "Complete-only trace"],
)
trace_comparison["Difference (full - complete)"] = (
    trace_comparison["Full trace"] - trace_comparison["Complete-only trace"]
)
display(trace_comparison)


In [ ]:
# Domain integrity and data-quality checks
source_ordered = log.sort_values([CASE_ID, SOURCE_ORDER], kind="mergesort")
source_timestamp_diffs = source_ordered.groupby(CASE_ID, sort=False)[TIMESTAMP].diff()
out_of_order_events = int(source_timestamp_diffs.lt(pd.Timedelta(0)).sum())
nat_event_count = int(log[TIMESTAMP].isna().sum())
selected_terminal_statuses = ["A_Pending", "A_Cancelled", "A_Denied"]

last_event_rows = log.groupby(CASE_ID, sort=False).tail(1).set_index(CASE_ID)
last_activity = last_event_rows[ACTIVITY]
last_a_by_case = (
    log.loc[log[ACTIVITY].astype("string").str.startswith("A_", na=False)]
    .groupby(CASE_ID, sort=False)
    .tail(1)
    .set_index(CASE_ID)[ACTIVITY]
)
last_o_by_case = (
    log.loc[log[ACTIVITY].astype("string").str.startswith("O_", na=False)]
    .groupby(CASE_ID, sort=False)
    .tail(1)
    .set_index(CASE_ID)[ACTIVITY]
)
case_outcomes = pd.DataFrame(
    {
        "Last activity": last_activity,
        "Last A activity": last_a_by_case,
        "Last O activity": last_o_by_case,
    }
)
case_outcomes["Terminal A status"] = case_outcomes["Last A activity"].where(
    case_outcomes["Last A activity"].isin(selected_terminal_statuses),
    "Incomplete",
)
case_outcomes["Last A activity"] = case_outcomes["Last A activity"].fillna("No A_* event")
case_outcomes["Last O activity"] = case_outcomes["Last O activity"].fillna("No O_* event")
terminal_a_status = case_outcomes["Terminal A status"]
has_selected_terminal_status = terminal_a_status.isin(selected_terminal_statuses)

quality_checks = pd.DataFrame(
    [
        ("Events with NaT timestamps", nat_event_count, f"{nat_event_count / len(log):.2%}"),
        ("Out-of-order timestamps in source case order", out_of_order_events, ""),
        ("Cases with a selected terminal A_* status", int(has_selected_terminal_status.sum()), f"{has_selected_terminal_status.mean():.2%}"),
        ("Incomplete / other terminal A_* status", int((~has_selected_terminal_status).sum()), f"{(~has_selected_terminal_status).mean():.2%}"),
    ],
    columns=["Check", "Count", "Share"],
)
display(Markdown("#### Timestamp and completion checks"))
display(quality_checks)

display(
    Markdown(
        "Terminal application status is the last `A_*` activity in each case, not necessarily "
        "the last event in the full process trace. `Incomplete` groups cases whose final "
        "application status is outside the three selected outcomes."
    )
)
terminal_status_counts = case_outcomes["Terminal A status"].value_counts().rename_axis("Terminal application status").reset_index(name="Cases")
display(Markdown("#### Terminal application status distribution"))
display(terminal_status_counts)

status_consistency = pd.crosstab(
    case_outcomes["Last A activity"],
    case_outcomes["Last O activity"],
    margins=True,
).rename_axis(index="Last application activity", columns="Last offer activity")
expected_offer_outcomes = [
    "O_Accepted",
    "O_Cancelled",
    "O_Refused",
    "No O_* event",
]
for outcome in expected_offer_outcomes:
    if outcome not in status_consistency.columns:
        status_consistency[outcome] = 0
extra_offer_outcomes = [
    column
    for column in status_consistency.columns
    if column not in expected_offer_outcomes and column != "All"
]
status_consistency = status_consistency[
    expected_offer_outcomes + extra_offer_outcomes + ["All"]
]
display(Markdown("#### Application and offer status cross-tabulation"))
display(status_consistency)

fraud_cases = log.groupby(CASE_ID, sort=False)[ACTIVITY].apply(
    lambda activities: activities.eq("W_Assess potential fraud").any()
)
fraud_case_ids = fraud_cases.index[fraud_cases]
fraud_outcomes = (
    case_outcomes.loc[fraud_case_ids, "Last activity"]
    .value_counts()
    .rename_axis("Final activity in fraud-assessment cases")
    .reset_index(name="Cases")
)
display(Markdown("#### Outcomes for cases with `W_Assess potential fraud`"))
display(fraud_outcomes)

terminal_chart_counts = case_outcomes["Terminal A status"].value_counts()
terminal_chart = pd.Series(
    {
        status: int(terminal_chart_counts.get(status, 0))
        for status in selected_terminal_statuses + ["Incomplete"]
    },
    name="Cases",
)
completed_case_ids = case_outcomes.index[has_selected_terminal_status]
top_final_events = (
    last_activity.loc[completed_case_ids]
    .value_counts()
    .head(10)
    .rename_axis("Final event in completed cases")
    .reset_index(name="Cases")
)
display(Markdown("#### Top 10 final events in cases with a selected terminal application status"))
display(top_final_events)

fraud_chart_counts = (
    case_outcomes.loc[fraud_case_ids, "Last activity"]
    .value_counts()
    .head(10)
    .sort_values()
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
terminal_chart.plot.bar(ax=axes[0], color="#4472C4")
axes[0].set(title="Terminal application outcomes", xlabel="Terminal outcome", ylabel="Cases")
axes[0].tick_params(axis="x", rotation=25)
fraud_chart_counts.plot.barh(ax=axes[1], color="#ED7D31")
axes[1].set(title="Final activities in fraud-assessment cases", xlabel="Cases", ylabel="Final activity")
fig.tight_layout()
plt.show()

In [ ]:
# Preprocessing and filtering for discovery
selected_terminal_statuses = ["A_Pending", "A_Cancelled", "A_Denied"]
discovery_case_ids = case_outcomes.index[
    case_outcomes["Terminal A status"].isin(selected_terminal_statuses)
]
discovery_log = log.loc[log[CASE_ID].isin(discovery_case_ids)].copy()
complete_lifecycle_log = discovery_log.loc[
    discovery_log[LIFECYCLE].astype("string").str.casefold().eq("complete")
].copy()

if discovery_log.empty:
    raise ValueError("No cases have a final application status of A_Pending, A_Cancelled, or A_Denied.")

def variant_count(frame):
    if frame.empty:
        return 0
    return int(frame.groupby(CASE_ID, sort=False)[ACTIVITY].agg(tuple).nunique())

filter_summary = pd.DataFrame(
    {
        "Full event log": [
            log[CASE_ID].nunique(),
            len(log),
            variant_count(log),
        ],
        "Selected terminal A_* cases (all lifecycle events)": [
            discovery_log[CASE_ID].nunique(),
            len(discovery_log),
            variant_count(discovery_log),
        ],
        "Selected cases (`complete` events only)": [
            complete_lifecycle_log[CASE_ID].nunique(),
            len(complete_lifecycle_log),
            variant_count(complete_lifecycle_log),
        ],
    },
    index=["Cases", "Events", "Variants"],
)
display(filter_summary)
display(
    Markdown(
        "Discovery uses **all lifecycle transitions** from cases whose final application status "
        "is `A_Pending`, `A_Cancelled`, or `A_Denied`, matching the original preprocessing intent. "
        "`complete_lifecycle_log` is retained as a separate optional slice; it does not overwrite "
        "the discovery log."
    )
)

In [ ]:
# Process discovery, BPMN export, external models, and Petri-net views
from itertools import product
import re
import tempfile

noise_thresholds = [0.0, 0.015, 0.03, 0.1]
dependency_thresholds = [0.5, 0.7]
and_thresholds = [0.5, 0.65]
loop_two_threshold = 0.5
discovered_models = {}

for noise_threshold in noise_thresholds:
    model_name = f"Inductive Miner, noise={noise_threshold:g}"
    discovered_models[model_name] = pm4py.discover_petri_net_inductive(
        discovery_log,
        noise_threshold=noise_threshold,
    )

for dependency_threshold, and_threshold in product(dependency_thresholds, and_thresholds):
    model_name = (
        f"Heuristics Miner, dependency={dependency_threshold:g}, "
        f"and={and_threshold:g}, loop2={loop_two_threshold:g}"
    )
    discovered_models[model_name] = pm4py.discover_petri_net_heuristics(
        discovery_log,
        dependency_threshold=dependency_threshold,
        and_threshold=and_threshold,
        loop_two_threshold=loop_two_threshold,
    )

def safe_filename(value):
    filename = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._")
    return filename or "process_model"

discovery_summary = []
for model_name, (net, initial_marking, final_marking) in discovered_models.items():
    bpmn_model = pm4py.convert_to_bpmn(net, initial_marking, final_marking)
    bpmn_path = DISCOVERED_BPMN_DIR / f"{safe_filename(model_name)}.bpmn"
    pm4py.write_bpmn(bpmn_model, str(bpmn_path))
    sound, diagnostics = pm4py.analysis.check_soundness(net, initial_marking, final_marking)
    discovery_summary.append(
        {
            "Model": model_name,
            "Places": len(net.places),
            "Transitions": len(net.transitions),
            "Arcs": len(net.arcs),
            "Sound": sound,
            "Soundness diagnostics": "OK" if sound else str(diagnostics),
            "BPMN export": str(bpmn_path.relative_to(DATA_DIR)),
        }
    )
display(pd.DataFrame(discovery_summary))

external_bpmn_files = sorted(
    path for path in EXTERNAL_BPMN_DIR.iterdir()
    if path.is_file() and path.suffix.casefold() == ".bpmn"
)
external_model_names = []
for bpmn_path in external_bpmn_files:
    bpmn_model = pm4py.read_bpmn(str(bpmn_path))
    discovered_models[f"External BPMN: {bpmn_path.stem}"] = pm4py.convert_to_petri_net(bpmn_model)
    external_model_names.append(f"External BPMN: {bpmn_path.stem}")

if external_model_names:
    display(pd.DataFrame({"Loaded external BPMN models": external_model_names}))
else:
    display(Markdown(f"No external BPMN models found in `{EXTERNAL_BPMN_DIR}`."))

display(Markdown("#### Discovered Petri-net structures"))
with tempfile.TemporaryDirectory(prefix="bpic2017_petri_nets_") as preview_directory:
    for model_index, (model_name, (net, initial_marking, final_marking)) in enumerate(discovered_models.items()):
        display(Markdown(f"**{model_name}**"))
        preview_path = Path(preview_directory) / f"model_{model_index}.png"
        pm4py.save_vis_petri_net(
            net,
            initial_marking,
            final_marking,
            str(preview_path),
            rankdir="LR",
        )
        display(Image(filename=str(preview_path)))


In [ ]:
# Final evaluation metrics and model comparison
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluation
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluation

evaluation_log = discovery_log
quality_metrics = []
for model_name, (net, initial_marking, final_marking) in discovered_models.items():
    fitness_result = pm4py.fitness_token_based_replay(
        evaluation_log, net, initial_marking, final_marking
    )
    fitness = float(fitness_result["log_fitness"])
    precision = float(
        pm4py.precision_token_based_replay(
            evaluation_log, net, initial_marking, final_marking
        )
    )
    generalization = float(
        generalization_evaluation.apply(
            evaluation_log, net, initial_marking, final_marking
        )
    )
    simplicity = float(simplicity_evaluation.apply(net))
    f1_score = (
        2 * fitness * precision / (fitness + precision)
        if fitness + precision
        else 0.0
    )
    quality_metrics.append(
        {
            "Model": model_name,
            "Replay Fitness": fitness,
            "Precision": precision,
            "F1-Score": f1_score,
            "Generalization": generalization,
            "Simplicity": simplicity,
            "Places": len(net.places),
            "Transitions": len(net.transitions),
            "Arcs": len(net.arcs),
        }
    )

results = pd.DataFrame(quality_metrics)
quality_columns = [
    "Replay Fitness",
    "Precision",
    "F1-Score",
    "Generalization",
    "Simplicity",
]

historical_csv_path = DATA_DIR / "processed" / "legacy" / "in_model_comparison.csv"
if not historical_csv_path.exists():
    raise FileNotFoundError(f"Archived comparison results are missing: {historical_csv_path}")
historical_csv_results = pd.read_csv(historical_csv_path)
historical_csv_results = historical_csv_results.loc[
    historical_csv_results["Model"].ne("Model")
].copy()
for column in historical_csv_results.columns[1:]:
    historical_csv_results[column] = pd.to_numeric(
        historical_csv_results[column], errors="raise"
    )
display(Markdown("#### All saved comparison results from the source project"))
display(historical_csv_results.reset_index(drop=True))

historical_results = pd.DataFrame(
    [
        ("Inductive Miner, all variants, noise=0", 1.000, 100.000, 0.194, 0.948, 0.628, 0.299, 0.023, 55, 87, 184),
        ("Inductive Miner, all variants, noise=0.2", 0.977, 0.000, 0.516, 0.946, 0.634, 0.347, 0.028, 46, 72, 152),
        ("Inductive Miner, valid end activities", 1.000, 99.318, 0.225, 0.947, 0.627, 0.292, 0.022, 59, 89, 192),
        ("Inductive Miner, valid end activities, noise=0.1", 0.979, 30.321, 0.424, 0.903, 0.644, 0.299, 0.029, 49, 67, 148),
        ("Inductive Miner, valid end activities, noise=0.03", 0.995, 67.447, 0.505, 0.906, 0.670, 0.302, 0.026, 60, 86, 182),
        ("Inductive Miner, valid end activities, noise=0.015", 1.000, 99.495, 0.228, 0.963, 0.626, 0.410, 0.032, 36, 61, 126),
        ("Heuristics Miner, dependency=0.5, and=0.5, loop2=0.5", 0.949, 0.000, 0.877, 0.911, 0.522, 0.333, 0.018, 42, 78, 175),
        ("Heuristics Miner, dependency=0.5, and=0.65, loop2=0.5", 0.954, 0.000, 0.877, 0.930, 0.522, 0.333, 0.018, 42, 78, 175),
        ("Heuristics Miner, dependency=0.7, and=0.5, loop2=0.5", 0.949, 0.000, 0.877, 0.945, 0.518, 0.351, 0.018, 40, 74, 167),
        ("Heuristics Miner, dependency=0.7, and=0.65, loop2=0.5", 0.950, 0.000, 0.877, 0.932, 0.520, 0.342, 0.018, 41, 76, 171),
    ],
    columns=[
        "Historical model",
        "Train fitness",
        "Test fit cases (%)",
        "Precision",
        "Generalization",
        "Simplicity (PM4Py)",
        "Visible/total transitions",
        "Cyclomatic simplicity",
        "Places",
        "Transitions",
        "Arcs",
    ],
)
display(Markdown(
    "#### Historical results retained from the original notebook\n"
    "These are previously recorded results, not recomputed in this run. Their original "
    "training and test splits may differ from the consistent filtered-log evaluation below."
))
display(historical_results)

plot_data = results.set_index("Model")[quality_columns]
fig, ax = plt.subplots(figsize=(max(14, 1.5 * len(plot_data)), 7))
plot_data.plot(kind="bar", ax=ax, width=0.82)
ax.set(
    title="Process-model quality metrics",
    xlabel="Discovered or imported model",
    ylabel="Score",
    ylim=(0, 1.12),
)
ax.tick_params(axis="x", rotation=35)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
ax.legend(title="Quality metric", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

display(Markdown("#### Final model comparison"))
display(
    results.style
    .format({column: "{:.3f}" for column in quality_columns})
    .highlight_max(subset=quality_columns, color="#C6EFCE")
    .set_caption("Recomputed model metrics; best value highlighted per quality metric")
)
